In [35]:
import pandas as pd

# Load the dataset
df = pd.read_excel('C:/Users/Lenevo/telecom-churn-analytics/telecom-churn-analytics/data/raw/Telco_customer_churn.xlsx')

# Quick inspection
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

Shape: (7043, 33)

Columns: ['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value', 'Churn Score', 'CLTV', 'Churn Reason']

First 5 rows:


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [36]:
print("Data Types:")
print(df.dtypes)


Data Types:
CustomerID               str
Count                  int64
Country                  str
State                    str
City                     str
Zip Code               int64
Lat Long                 str
Latitude             float64
Longitude            float64
Gender                   str
Senior Citizen           str
Partner                  str
Dependents               str
Tenure Months          int64
Phone Service            str
Multiple Lines           str
Internet Service         str
Online Security          str
Online Backup            str
Device Protection        str
Tech Support             str
Streaming TV             str
Streaming Movies         str
Contract                 str
Paperless Billing        str
Payment Method           str
Monthly Charges      float64
Total Charges         object
Churn Label              str
Churn Value            int64
Churn Score            int64
CLTV                   int64
Churn Reason             str
dtype: object


In [37]:
print("\nMissing Values:")
print(df.isnull().sum())



Missing Values:
CustomerID              0
Count                   0
Country                 0
State                   0
City                    0
Zip Code                0
Lat Long                0
Latitude                0
Longitude               0
Gender                  0
Senior Citizen          0
Partner                 0
Dependents              0
Tenure Months           0
Phone Service           0
Multiple Lines          0
Internet Service        0
Online Security         0
Online Backup           0
Device Protection       0
Tech Support            0
Streaming TV            0
Streaming Movies        0
Contract                0
Paperless Billing       0
Payment Method          0
Monthly Charges         0
Total Charges           0
Churn Label             0
Churn Value             0
Churn Score             0
CLTV                    0
Churn Reason         5174
dtype: int64


In [38]:
print("\nChurn Distribution:")
print(df['Churn Label'].value_counts())



Churn Distribution:
Churn Label
No     5174
Yes    1869
Name: count, dtype: int64


In [39]:
print("\nChurn %:")
print(df['Churn Label'].value_counts(normalize=True).mul(100).round(2))


Churn %:
Churn Label
No     73.46
Yes    26.54
Name: proportion, dtype: float64


In [40]:
# Check only columns with missing values
missing = df.isnull().sum()
missing_cols = missing[missing > 0]
print("Columns with missing values:")
print(missing_cols)
print(f"\nTotal columns with missing values: {len(missing_cols)}")


Columns with missing values:
Churn Reason    5174
dtype: int64

Total columns with missing values: 1


## Data Quality Assessment

Dataset contains 7,043 rows and 33 columns.

**Missing Values:**
- `Churn Reason` has 5,174 missing values — this is expected and logical.
  Only churned customers (1,869) have a churn reason. Retained customers
  naturally have no churn reason. No imputation required.

**Conclusion:** Dataset is clean with no unexpected missing values.
All other 32 columns are complete.

In [41]:
# Check data types
print(df.dtypes)


CustomerID               str
Count                  int64
Country                  str
State                    str
City                     str
Zip Code               int64
Lat Long                 str
Latitude             float64
Longitude            float64
Gender                   str
Senior Citizen           str
Partner                  str
Dependents               str
Tenure Months          int64
Phone Service            str
Multiple Lines           str
Internet Service         str
Online Security          str
Online Backup            str
Device Protection        str
Tech Support             str
Streaming TV             str
Streaming Movies         str
Contract                 str
Paperless Billing        str
Payment Method           str
Monthly Charges      float64
Total Charges         object
Churn Label              str
Churn Value            int64
Churn Score            int64
CLTV                   int64
Churn Reason             str
dtype: object


In [42]:
print(df['Total Charges'].dtype)
print(df['Monthly Charges'].dtype)
print(df['Churn Value'].dtype)


object
float64
int64


In [43]:
# Check what's causing Total Charges to be object type
print(df['Total Charges'].unique()[:20])
print("\nAny spaces or blanks?")
print(df['Total Charges'].str.strip().eq('').sum())


[108.15 151.65 820.5 3046.05 5036.3 528.35 39.65 20.15 4749.15 30.2 1093.1
 316.9 3549.25 1105.4 144.15 1426.4 633.3 1752.55 857.25 79.35]

Any spaces or blanks?
11


In [44]:
# Investigate the 11 problematic rows before fixing them
mask = df['Total Charges'].str.strip().eq('')
print(df[mask][['CustomerID', 'Tenure Months', 'Monthly Charges', 'Total Charges', 'Churn Label']])


      CustomerID  Tenure Months  Monthly Charges Total Charges Churn Label
2234  4472-LVYGI              0            52.55                        No
2438  3115-CZMZD              0            20.25                        No
2568  5709-LVOEQ              0            80.85                        No
2667  4367-NUYAO              0            25.75                        No
2856  1371-DWPAZ              0            56.05                        No
4331  7644-OMVMY              0            19.85                        No
4687  3213-VVOLG              0            25.35                        No
5104  2520-SGTTA              0            20.00                        No
5719  2923-ARZLG              0            19.70                        No
6772  4075-WKNIU              0            73.35                        No
6840  2775-SEFEE              0            61.90                        No


## Handling Missing Values in Total Charges

**Issue:** 11 rows found with blank `Total Charges` values.

**Investigation:** All 11 rows have `Tenure Months = 0` and `Churn Label = No`.

**Conclusion:** These are brand new customers who have not completed 
their first billing cycle yet. They have no billing history and no 
churn behaviour to analyse.

**Decision:** Drop these 11 rows.
- They represent only 0.15% of the dataset (11 out of 7,043 rows)
- Retaining them would introduce noise into revenue and churn analysis
- A customer with 0 tenure has not had enough interaction with the 
  service to exhibit any churn signals

**Result:** Dataset reduced from 7,043 to 7,032 rows. No analytical 
impact.

In [45]:
# Drop rows where Total Charges is blank (Tenure = 0 customers)
df = df[df['Tenure Months'] > 0].reset_index(drop=True)

# Verify
print("Rows before:", 7043)
print("Rows after:", len(df))
print("Rows dropped:", 7043 - len(df))

# Convert Total Charges to numeric (already clean after dropping tenure=0 rows)
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')
print("\nTotal Charges dtype:", df['Total Charges'].dtype)
print("Remaining NaN values:", df['Total Charges'].isnull().sum())

Rows before: 7043
Rows after: 7032
Rows dropped: 11

Total Charges dtype: float64
Remaining NaN values: 0


## Feature Engineering

Creating new analytical columns to support deeper churn analysis.
These features are not in the original dataset but are derived from
existing columns to add business meaning.

In [46]:
# 1. Tenure Band — groups customers by loyalty stage
df['Tenure Band'] = pd.cut(df['Tenure Months'],
                            bins=[0, 12, 24, 48, 60, 72],
                            labels=['0-12 Months', '13-24 Months',
                                    '25-48 Months', '49-60 Months', '61+ Months'])

# 2. Revenue Segment — splits customers into Low/Mid/High spenders
df['Revenue Segment'] = pd.qcut(df['Monthly Charges'], q=3,
                                  labels=['Low', 'Mid', 'High'])

# 3. Service Count — how many add-on services each customer has
service_cols = ['Phone Service', 'Online Security', 'Online Backup',
                'Device Protection', 'Tech Support',
                'Streaming TV', 'Streaming Movies']
df['Service Count'] = df[service_cols].apply(
    lambda row: sum(row == 'Yes'), axis=1)

# 4. Contract Risk Score — month-to-month = highest risk
contract_map = {'Month-to-month': 3, 'One year': 2, 'Two year': 1}
df['Contract Risk'] = df['Contract'].map(contract_map)

# Verify new columns
print("New columns added:")
print(df[['Tenure Band', 'Revenue Segment', 'Service Count', 'Contract Risk']].head(10))


New columns added:
    Tenure Band Revenue Segment  Service Count  Contract Risk
0   0-12 Months             Mid              3              3
1   0-12 Months             Mid              1              3
2   0-12 Months            High              4              3
3  25-48 Months            High              5              3
4  49-60 Months            High              5              3
5   0-12 Months             Mid              3              3
6   0-12 Months             Low              2              3
7   0-12 Months             Low              1              3
8  25-48 Months            High              4              3
9   0-12 Months             Low              1              3


## Saving Cleaned Dataset

Exporting the cleaned and feature-engineered dataset to data/processed/
for use in EDA analysis and Power BI dashboard.

In [47]:
# Save cleaned dataset
output_path = 'C:/Users/Lenevo/telecom-churn-analytics/telecom-churn-analytics/data/processed/telco_churn_clean.csv'

df.to_csv(output_path, index=False)

print("Dataset saved successfully!")
print(f"Shape: {df.shape}")
print(f"Location: {output_path}")

PermissionError: [Errno 13] Permission denied: 'C:/Users/Lenevo/telecom-churn-analytics/telecom-churn-analytics/data/processed/telco_churn_clean.csv'

In [ ]:
# Final dataset summary
print("=" * 50)
print("CLEANED DATASET SUMMARY")
print("=" * 50)
print(f"Total Rows:        {df.shape[0]}")
print(f"Total Columns:     {df.shape[1]}")
print(f"\nChurn Distribution:")
print(df['Churn Label'].value_counts())
print(f"\nChurn Rate:        {round(df['Churn Value'].mean() * 100, 2)}%")
print(f"\nAvg Monthly Charges:  ${round(df['Monthly Charges'].mean(), 2)}")
print(f"Avg Total Charges:    ${round(df['Total Charges'].mean(), 2)}")
print(f"\nNew Features Added:")
print("- Tenure Band")
print("- Revenue Segment")
print("- Service Count")
print("- Contract Risk")
print("=" * 50)

CLEANED DATASET SUMMARY
Total Rows:        7032
Total Columns:     37

Churn Distribution:
Churn Label
No     5163
Yes    1869
Name: count, dtype: int64

Churn Rate:        26.58%

Avg Monthly Charges:  $64.8
Avg Total Charges:    $2283.3

New Features Added:
- Tenure Band
- Revenue Segment
- Service Count
- Contract Risk
